In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
def run_ols_regression(gasreg, data):
    """
    Fit an OLS regression model using daily HDD/CDDs for a given gasreg
    as independent variables and daily multiplicative price differences
    from the annual average price for that gasreg.
    """
    data['month'] = data.index.get_level_values('month')
    month_dummies = (
        pd.get_dummies(data['month'], prefix='month')
        .drop(columns='month_5')
        .astype(float)
    )
    
    degree_day_columns = [f'{gasreg}_{dd_var}' for dd_var in ['cdd', 'hdd']]
    temperature_variable = data[degree_day_columns].astype(float)
    
    X = pd.concat([temperature_variable, month_dummies], axis=1)
    X = sm.add_constant(X)
    y = data[f"{gasreg}_price_diff"].values.astype(float)
    
    # The lines below represent an ordinary least-squares regression using a
    # heteroskedasticity- and autocorrelation-consistent (HAC) estimator,
    # which ensures that the standard errors calculated in the regression
    # are robust to heteroskedastic and autocorrelated residuals (both
    # common when working with time series data).
    # The maxlags value represents the maximum number of timesteps
    # (in this case days) across which the estimator adjusts for auto-
    # correlation. The value is calculated using the Stock and Watson rule-of-thumb:
    # number of lags = 0.75 * (number of observations)**(1/3)
    # (see https://www.econometrics-with-r.org/15.4-hac-standard-errors.html)
    maxlags = int(round(0.75 * len(data)**(1/3)))
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})

    return model

def apply_regression_model(gasreg, model, data):
    """
    Apply an OLS regression model to predict daily multiplicative price
    differences from daily HDD/CDDs for a given gasreg.
    """
    data['month'] = data.index.get_level_values('month')
    month_dummies = (
        pd.get_dummies(data['month'], prefix='month')
        .drop(columns='month_5')
        .astype(float)
    )
    
    degree_day_columns = [f'{gasreg}_{dd_var}' for dd_var in ['cdd', 'hdd']]
    temperature_variable = data[degree_day_columns].astype(float)

    X_test = pd.concat([temperature_variable, month_dummies], axis=1)
    X_test = sm.add_constant(X_test)
    y_pred = model.predict(X_test)

    return y_pred

In [3]:
# Get daily HDD/CDDs and prices for each gasreg
gasreg_data = pd.read_csv(
    Path('inputs', 'gasreg_regression_data.csv'),
    index_col=['year', 'month', 'day']
)
gasreg_degree_days = (
    gasreg_data[[col for col in gasreg_data.columns if 'dd' in col]]
    .copy()
)
gasreg_prices = (
    gasreg_data[[col for col in gasreg_data.columns if 'price' in col]]
    .copy()
)
gasreg_prices.columns = [col.replace('_price', '') for col in gasreg_prices.columns]

# Calculate average annual prices for each gasreg
# and the daily deviations (log of the multiplicative difference)
# from the annual average price
gasreg_annual_average_prices = (
    gasreg_prices.groupby(gasreg_prices.index.get_level_values('year'))
    .transform('mean')
)
gasreg_log_mult_diffs = np.log(gasreg_prices) - np.log(gasreg_annual_average_prices)

# Replace prices with multiplicative price differences
gasreg_data = pd.concat([
    gasreg_data.loc[:, ~gasreg_data.columns.str.contains('_price')],
    gasreg_log_mult_diffs.add_suffix('_price_diff')
], axis=1)

In [4]:
# Fit the regression model using data from 2014-2023 and test on 2024
train_years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
test_years = [2024]

gasreg_model_params = {}
for gasreg in gasreg_prices.columns:
    data = gasreg_data.filter(like=gasreg)
    train_data = data.loc[data.index.get_level_values('year').isin(train_years)].copy()
    model = run_ols_regression(gasreg, train_data)

    # Save model parameters (coefficients, intercepts, and fixed effects)
    gasreg_model_params[gasreg] = (
        model.params
        .rename({
            f"{gasreg}_cdd": 'cdd',
            f"{gasreg}_hdd": 'hdd'
        })
    )
    
    # Calculate and print relevant statistics to assess model effectiveness
    test_data = data.loc[~data.index.get_level_values('year').isin(train_years)].copy()
    y_pred = apply_regression_model(gasreg, model, test_data)
    y_true = test_data[f"{gasreg}_price_diff"]
    
    test_r2 = r2_score(y_true, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_true, y_pred))    

    print(f"{gasreg}")
    print(f"Train R2: {model.rsquared.round(3)}")
    print(f"Train Adj R2: {model.rsquared_adj.round(3)}")
    print(f"Test R2: {test_r2.round(3)}")
    print(f"Test RMSE: {test_rmse.round(3)}")
    print(f"Standard deviation of test data: {round(np.std(y_true), 3)}")
    print(f"Coefficients and p-values:\n{dict(zip(model.params[1:3].round(3), model.pvalues[1:3].round(3)))}")
    print('')    

California
Train R2: 0.185
Train Adj R2: 0.183
Test R2: 0.277
Test RMSE: 0.321
Standard deviation of test data: 0.377
Coefficients and p-values:
{0.048: 0.0, 0.032: 0.0}

East_North_Central
Train R2: 0.132
Train Adj R2: 0.129
Test R2: 0.21
Test RMSE: 0.323
Standard deviation of test data: 0.363
Coefficients and p-values:
{0.031: 0.0, 0.022: 0.0}

East_South_Central
Train R2: 0.147
Train Adj R2: 0.144
Test R2: 0.197
Test RMSE: 0.299
Standard deviation of test data: 0.334
Coefficients and p-values:
{0.014: 0.028, 0.018: 0.0}

Mid_Atlantic
Train R2: 0.419
Train Adj R2: 0.417
Test R2: 0.495
Test RMSE: 0.316
Standard deviation of test data: 0.445
Coefficients and p-values:
{0.096: 0.0, 0.052: 0.0}

Mountain
Train R2: 0.183
Train Adj R2: 0.18
Test R2: 0.305
Test RMSE: 0.373
Standard deviation of test data: 0.447
Coefficients and p-values:
{0.042: 0.001, 0.027: 0.0}

New_England
Train R2: 0.59
Train Adj R2: 0.588
Test R2: 0.635
Test RMSE: 0.38
Standard deviation of test data: 0.628
Coefficien

In [5]:
# Reformat regression parameters for ReEDS
regression_params = (
    pd.concat(gasreg_model_params, axis=1)
    .rename({
        'const': 'alpha',
        'cdd': 'beta_CDD',
        'hdd': 'beta_HDD',
        'month_1': 'alpha_JAN',
        'month_2': 'alpha_FEB',
        'month_3': 'alpha_MAR',
        'month_4': 'alpha_APR',
        'month_6': 'alpha_JUN',
        'month_7': 'alpha_JUL',
        'month_8': 'alpha_AUG',
        'month_9': 'alpha_SEP',
        'month_10': 'alpha_OCT',
        'month_11': 'alpha_NOV',
        'month_12': 'alpha_DEC'
    })
)
regression_params.loc['alpha_MAY'] = 0

In [6]:
# Export
outpath = Path('outputs', 'gasreg_degree_day_price_mult_regression_params.csv')
outpath.parent.mkdir(parents=True, exist_ok=True)
(
    regression_params.loc[[
        'beta_CDD',
        'beta_HDD',
        'alpha',
        'alpha_JAN',
        'alpha_FEB',
        'alpha_MAR',
        'alpha_APR',
        'alpha_MAY',
        'alpha_JUN',
        'alpha_JUL',
        'alpha_AUG',
        'alpha_SEP',
        'alpha_OCT',
        'alpha_NOV',
        'alpha_DEC'
    ]]
    .rename_axis('param')
    .round(3)
    .sort_index(axis=1)
    .to_csv(outpath)
)